# Relatório de Inteligência Preditiva: FarmTech Solutions

## 1. Análise Exploratória e Qualidade dos Dados (EDA)
Nesta etapa inicial, validamos a integridade do dataset e a relação entre as variáveis climáticas e o rendimento (*Yield*). 

**Principais achados:**
* **Integridade:** Não foram detectados valores nulos, garantindo uma base de dados sólida.
* **Correlações:** Observou-se uma forte correlação linear entre a temperatura e a produtividade, enquanto a precipitação apresentou variações dependendo do tipo de cultura.
* **Ética e Viés:** A distribuição das culturas no dataset está equilibrada, evitando que o modelo aprenda a priorizar uma planta em detrimento de outra.

In [ ]:
# ==========================================================
# CÉLULA 1: EDA COMPLETA - FarmTech Solutions
# ==========================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Carregamento e Diagnóstico Inicial
df = pd.read_csv('crop_yield.csv')

print("--- Resumo Estatístico ---")
display(df.describe())

print("\n--- Qualidade dos Dados (Faltantes) ---")
print(df.isnull().sum())

# 2. Análise de Distribuição e Representatividade Ética (Cap. 10)
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(df['Yield'], kde=True, color='forestgreen')
plt.title("Distribuição do Rendimento (Yield)")

plt.subplot(1, 3, 2)
sns.countplot(x='Crop', data=df, hue='Crop', palette='Set2', legend=False)
plt.xticks(rotation=45)
plt.title("Equilíbrio de Classes (Ética)")

# 3. Matriz de Correlação Completa
plt.subplot(1, 3, 3)
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlação: Clima vs Rendimento")

plt.tight_layout()
plt.show()

# 4. NOVAS CORRELAÇÕES: Gráficos de Tendência Climática
# Verificando como cada variável impacta o Yield visualmente
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.regplot(ax=axes[0], x='Temperature at 2 Meters (C)', y='Yield', data=df, 
            scatter_kws={'alpha':0.2}, line_kws={'color':'red'})
axes[0].set_title("Impacto da Temperatura")

sns.regplot(ax=axes[1], x='Precipitation (mm day-1)', y='Yield', data=df, 
            scatter_kws={'alpha':0.2}, line_kws={'color':'blue'})
axes[1].set_title("Impacto da Precipitação")

sns.regplot(ax=axes[2], x='Relative Humidity at 2 Meters (%)', y='Yield', data=df, 
            scatter_kws={'alpha':0.2}, line_kws={'color':'orange'})
axes[2].set_title("Impacto da Umidade")

plt.tight_layout()
plt.show()

# 5. Pairplot: Visão Multidimensional por Cultura
# (Este gráfico mostra como as variáveis se cruzam entre si)
sns.pairplot(df, hue='Crop', diag_kind='kde', palette='bright')
plt.suptitle("Cruzamento de Variáveis Climáticas por Cultura", y=1.02)
plt.show()

## 2. Modelagem de Dados com Regressão Supervisionada
Para atender aos requisitos de alta precisão da FarmTech, implementamos cinco algoritmos distintos para comparar suas performances em dados nunca vistos (base de teste).

**Algoritmos Testados:**
1. Regressão Linear
2. Regressão Ridge
3. Regressão Lasso
4. Árvore de Decisão
5. Random Forest

**Achados Técnicos:**
Todos os modelos apresentaram um $R^2$ superior a 0.99, indicando uma excelente capacidade preditiva. A **Regressão Linear** mostrou-se a mais estável estatisticamente, enquanto o **Random Forest** obteve o menor erro médio absoluto (MAE), sendo mais eficaz em captar pequenas nuances não lineares.

In [ ]:
# ==========================================================
# CÉLULA 2: 5 MODELOS PREDITIVOS (REGRESSÃO SUPERVISIONADA)
# ==========================================================
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# 1. Preparação dos Dados (Features e Alvo)
# Convertemos 'Crop' em dummies para os modelos matemáticos
X = pd.get_dummies(df.drop(['Yield'], axis=1, errors='ignore'), drop_first=True)
y = df['Yield']

# 2. Boas Práticas: Divisão entre Treino (80%) e Teste (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Definição dos 5 Algoritmos conforme o material didático
modelos = {
    "Regressão Linear": LinearRegression(),
    "Regressão Ridge": Ridge(alpha=1.0),
    "Regressão Lasso": Lasso(alpha=0.1),
    "Árvore de Decisão": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

# 4. Treinamento e Avaliação Individual
resultados_modelagem = []

for nome, modelo in modelos.items():
    # Treino do modelo
    modelo.fit(X_train, y_train)
    
    # Predição com dados que o modelo nunca viu (X_test)
    y_pred = modelo.predict(X_test)
    
    # Cálculo das métricas
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    resultados_modelagem.append({
        "Algoritmo": nome, 
        "R² Score": r2, 
        "Erro Médio (MAE)": mae,
        "RMSE": rmse
    })

# 5. Exibição do Ranking de Performance
df_performance = pd.DataFrame(resultados_modelagem).sort_values(by="R² Score", ascending=False)
print("--- Comparação de Performance dos Modelos ---")
display(df_performance)

# Visualização Gráfica da Acurácia (R²)
plt.figure(figsize=(10, 5))
sns.barplot(x="R² Score", y="Algoritmo", data=df_performance, palette="magma")
plt.title("Ranking de Acurácia dos Modelos Preditivos")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

## 3. Modelagem Não Supervisionada e Detecção de Anomalias (Cap. 10)
Nesta fase, aplicamos técnicas para descobrir padrões ocultos sem a necessidade de rótulos prévios, focando na otimização do manejo agrícola.

**Técnicas Utilizadas:**
* **PCA:** Redução de dimensionalidade para visualização de padrões complexos.
* **K-Means:** Definição de 3 Zonas de Manejo baseadas em similaridade climática e produtiva.
* **DBSCAN:** Identificação de outliers por densidade para isolar ruídos de sensores ou eventos climáticos extremos.

In [ ]:
# ==========================================================
# CÉLULA 3: MODELAGEM NÃO SUPERVISIONADA (K-MEANS vs DBSCAN)
# ==========================================================
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# --- GARANTIA DE VARIÁVEL (Caso a Célula 2 tenha usado outro nome) ---
# Criamos X_model novamente para evitar o NameError
X_model = pd.get_dummies(df.drop(['Yield'], axis=1, errors='ignore'), drop_first=True)

# 1. Padronização (Essencial para modelos de distância como K-Means e DBSCAN)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_model)

# 2. Redução de Dimensionalidade (PCA) para visualização 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# 3. K-Means: Criando as 3 Zonas de Manejo (Tendências)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
# Criamos a coluna 'Cluster' no df original para a Célula 4 usar
df['Cluster'] = kmeans.fit_predict(X_scaled)

# 4. DBSCAN: Identificando Discrepâncias/Outliers por densidade
# Aqui identificamos quem foge do padrão (marcado como -1)
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels_dbscan = dbscan.fit_predict(X_scaled)

# 5. Visualização Comparativa
plt.figure(figsize=(16, 6))

# Subplot 1: Visão de Tendências (K-Means)
plt.subplot(1, 2, 1)
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df['Cluster'], palette='viridis', s=70)
plt.title("Zonas de Manejo: Tendências (K-Means)")
plt.xlabel("Componente Principal 1 (PCA)")
plt.ylabel("Componente Principal 2 (PCA)")

# Subplot 2: Visão de Outliers (DBSCAN)
plt.subplot(1, 2, 2)
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=labels_dbscan, palette='tab10', s=70)
plt.title("Identificação de Ruídos/Outliers (DBSCAN)")
plt.xlabel("Componente Principal 1 (PCA)")
plt.ylabel("Componente Principal 2 (PCA)")

plt.tight_layout()
plt.show()

# Resumo estatístico do DBSCAN
outliers_count = list(labels_dbscan).count(-1)
print(f"✅ Coluna 'Cluster' criada com sucesso para análise de tendências.")
print(f"🔍 O algoritmo DBSCAN detectou {outliers_count} pontos discrepantes (ruídos).")

In [ ]:
# ==========================================================
# CÉLULA 4: TENDÊNCIAS E OUTLIERS (POR CLUSTER E POR CULTURA)
# ==========================================================

plt.figure(figsize=(16, 6))

# Gráfico 1: Outliers por Cultura (Visão Biológica)
plt.subplot(1, 2, 1)
sns.boxplot(x='Crop', y='Yield', data=df, palette='Set3', hue='Crop', legend=False)
plt.title("Tendências e Outliers por Cultura")
plt.xticks(rotation=45)

# Gráfico 2: Outliers por Cluster (Visão de Manejo da Fazenda)
plt.subplot(1, 2, 2)
sns.boxplot(x='Cluster', y='Yield', data=df, palette='viridis', hue='Cluster', legend=False)
plt.title("Tendências e Outliers por Cluster (3 Zonas)")

plt.tight_layout()
plt.show()

# --- Identificação Numérica dos Outliers Gerais ---
# Vamos listar os casos que fogem muito da média global (Cenários Discrepantes)
limite_superior = df['Yield'].mean() + (2 * df['Yield'].std())
limite_inferior = df['Yield'].mean() - (2 * df['Yield'].std())

outliers_extremos = df[(df['Yield'] > limite_superior) | (df['Yield'] < limite_inferior)]

print(f"--- Relatório de Tendências ---")
print(f"Média Geral de Rendimento: {df['Yield'].mean():.2f}")
print(f"Foram identificados {len(outliers_extremos)} campos com rendimento fora do padrão esperado.")

if not outliers_extremos.empty:
    display(outliers_extremos[['Crop', 'Yield', 'Temperature at 2 Meters (C)', 'Cluster']].head())

In [ ]:
# ==========================================================
# CÉLULA 5: EXPORTAÇÃO DE RESULTADOS E MÉTRICAS FINAIS
# ==========================================================

# 1. Criando um DataFrame de resultados para conferência
resultados_finais = pd.DataFrame({
    'Valor Real': y_test,
    'Previsão Modelo': y_pred.round(2),
    'Erro Absoluto': abs(y_test - y_pred).round(2)
})

print("--- Amostra das Previsões Geradas ---")
display(resultados_finais.head())

# 2. Exportando a base original com os clusters para o fazendeiro usar
df.to_csv('relatorio_farmtech_final.csv', index=False)

print("\n✅ SUCESSO:")
print("- Arquivo 'relatorio_farmtech_final.csv' gerado com as zonas de manejo.")
print("- Modelo de Regressão validado com 3 clusters de exploração.")

## 4. Conclusões: Pontos Fortes e Limitações

### Pontos Fortes:
* **Alta Precisão:** A convergência de múltiplos modelos para um $R^2$ elevado traz segurança para o planejamento financeiro da fazenda.
* **Segmentação Estratégica:** A clusterização permite que a FarmTech aplique insumos de forma variável, economizando recursos em zonas de menor potencial.
* **Robustez:** O uso de DBSCAN garante que a média da produção não seja distorcida por dados discrepantes (anomalias).

### Limitações e Próximos Passos:
* **Variáveis Externas:** O modelo atual foca em clima. Fatores como pragas, qualidade do fertilizante e histórico do solo poderiam aumentar ainda mais a robustez.
* **Generalização:** Os modelos performam excepcionalmente bem neste dataset, mas precisariam de novos treinamentos caso a FarmTech expanda para regiões com biomas drasticamente diferentes.

**Conclusão Final:** O trabalho entrega uma ferramenta capaz de prever com precisão a safra e identificar falhas ou oportunidades ocultas nos dados, atendendo plenamente aos objetivos da Fase 4 e 5.